In [14]:
import sympy as sp
import numpy as np
import sys

def newton_method():
    print("=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP TIẾP TUYẾN ===")
    print("(Mẹo: Gõ hàm số cần ghi rõ dấu nhân, ví dụ: x**3 + 3*x**2 - 1)\n")
    
    try:
        # 1. Nhập dữ liệu
        raw_f = input("Nhập hàm số f(x): ")
        f_input = raw_f.replace('^', '**')
        
        a = float(input("Nhập điểm a: "))
        b = float(input("Nhập điểm b: "))
        
        if a > b:
            a, b = b, a
            
        epsilon = float(input("Nhập sai số epsilon (VD: 1e-7): "))
        precision = int(input("Nhập số chữ số làm tròn (precision): "))

        # 2. Xử lý toán học
        x = sp.Symbol('x')
        f = sp.sympify(f_input)
        df = sp.diff(f, x)
        d2f = sp.diff(df, x)

        f_lam = sp.lambdify(x, f, 'numpy')
        df_lam = sp.lambdify(x, df, 'numpy')
        d2f_lam = sp.lambdify(x, d2f, 'numpy')

        X_vals = np.linspace(a, b, 1000)
        df_vals = df_lam(X_vals)
        d2f_vals = d2f_lam(X_vals)

        # 3. Kiểm tra các điều kiện an toàn
        fa, fb = f_lam(a), f_lam(b)
        
        if np.isclose(fa, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm của phương trình chính là tại mút a = {a}")
            return
        if np.isclose(fb, 0):
            print(f"\n=> KẾT QUẢ NGAY LẬP TỨC: Nghiệm của phương trình chính là tại mút b = {b}")
            return
            
        if fa * fb > 0:
            print(f"\n[!] LỖI: f({a}) = {fa:.4f} và f({b}) = {fb:.4f} cùng dấu. Không đảm bảo có nghiệm trên đoạn này.")
            return

        if np.any(df_vals > 0) and np.any(df_vals < 0):
            print("\n[!] LỖI: Đạo hàm f'(x) đổi dấu (lúc tăng lúc giảm) trên [a, b]. Vui lòng thu hẹp khoảng cách giữa a và b.")
            return
            
        if np.any(d2f_vals > 0) and np.any(d2f_vals < 0):
            print("\n[!] LỖI: Đạo hàm f''(x) đổi dấu (đồ thị đổi chiều lõm) trên [a, b]. Vui lòng thu hẹp khoảng cách giữa a và b.")
            return

        m1 = np.min(np.abs(df_vals))
        if np.isclose(m1, 0, atol=1e-10):
            print("\n[!] LỖI CHÍ MẠNG: Tồn tại điểm làm cho f'(x) = 0 trên đoạn [a, b]. Chương trình sẽ bị lỗi chia cho 0. Vui lòng thu hẹp khoảng nghiệm.")
            return

        M2 = np.max(np.abs(d2f_vals))
        x0 = a if f_lam(a) * d2f_lam(a) > 0 else b

        # Xuất kết quả
        print("\n" + "="*50)
        print("### 1. Kiểm tra điều kiện và Chọn điểm xuất phát")
        print(f"* **Khoảng phân li nghiệm:** $f({a}) = {fa:.{precision}f}$, $f({b}) = {fb:.{precision}f} \\Rightarrow f({a})f({b}) < 0$")
        print(f"* **Điều kiện đạo hàm:** $f'(x) = {sp.latex(df)}$ và $f''(x) = {sp.latex(d2f)}$ xác định dấu không đổi.")
        print(f"* **Chọn $x_0$:** Vì $f({x0}) \\cdot f''({x0}) > 0$, ta chọn $x_0 = {x0}$.")
        
        print("\n### 2. Xây dựng công thức lặp và Đánh giá sai số")
        print(f"* **Các hằng số:** $m_1 = \\min |f'(x)| = {m1:.{precision}f}$, $M_2 = \\max |f''(x)| = {M2:.{precision}f}$")
        print(f"* **Công thức lặp:**")
        print(f"  $$x_n = x_{{n-1}} - \\frac{{f(x_{{n-1}})}}{{f'(x_{{n-1}})}} = x_{{n-1}} - \\frac{{{sp.latex(f)}}}{{{sp.latex(df)}}}$$")

        # TÍNH TOÁN DỰ ĐOÁN TIÊN NGHIỆM TẠI ĐÂY
        x1_est = x0 - f_lam(x0) / df_lam(x0)
        q = (M2 / (2 * m1)) * abs(x1_est - x0)
        
        # Đã đổi định dạng q sang .g để tránh in ra 0.000
        print("* **Dự đoán số bước lặp (Tiên nghiệm):**")
        print(f"  Tính $q = \\frac{{M_2}}{{2m_1}} |x_1 - x_0| = {q:.{precision}g}$")
        if q < 1:
            P = epsilon * M2 / (2 * m1)
            if P < 1:
                n_est = np.ceil(np.log2(np.log(P) / np.log(q)))
                print(f"  $\\Rightarrow$ Để đạt sai số $\\epsilon = {epsilon:.1e}$, cần lặp tối đa: $n \\ge {int(n_est)}$ bước.")
            else:
                print("  $\\Rightarrow$ Nghiệm xấp xỉ đã rất sát, có thể đạt sai số ngay trong 1-2 bước đầu.")
        else:
            print("  $\\Rightarrow$ Do $q \\ge 1$, không thể ước lượng số bước lặp bằng công thức tiên nghiệm cơ bản (nhưng thuật toán vẫn có thể hội tụ).")

        print("* **Điều kiện dừng thực tế (Hậu nghiệm):**")
        print(f"  $$\\Delta_n = \\frac{{M_2}}{{2m_1}} |x_n - x_{{n-1}}|^2 < {epsilon:.1e}$$")

        # 4. Vòng lặp
        history = []
        xn = x0
        n = 1
        
        x_prev = xn
        fx_prev = f_lam(x_prev)
        dfx_prev = df_lam(x_prev)
        xn = x_prev - fx_prev / dfx_prev
        
        while True:
            if np.isclose(dfx_prev, 0, atol=1e-15):
                print(f"\n[!] LỖI: Tại bước lặp thứ {n}, giá trị đạo hàm tiến quá sát 0. Thuật toán bị vỡ.")
                break

            # Tính sai số (Hậu nghiệm)
            err = (M2 / (2 * m1)) * abs(xn - x_prev)**2
                
            history.append({
                'n': n,
                'x_prev': x_prev,
                'xn': xn,
                'fx_prev': fx_prev,
                'dfx_prev': dfx_prev,
                'err': err
            })
            
            if err < epsilon:
                break
                
            n += 1
            x_prev = xn
            fx_prev = f_lam(x_prev)
            dfx_prev = df_lam(x_prev)
            xn = x_prev - fx_prev / dfx_prev

        total_steps = len(history)

        print("\n### 3. Chi tiết các bước lặp")
        print("**Hai bước lặp đầu tiên:**")
        for i in range(min(2, total_steps)):
            step = history[i]
            print(f"* **$n={step['n']}$:**")
            print(f"  $$x_{step['n']} = {step['x_prev']:.{precision}f} - \\frac{{{step['fx_prev']:.{precision}f}}}{{{step['dfx_prev']:.{precision}f}}} = {step['xn']:.{precision}f}$$")
            # Dùng định dạng .g cho sai số
            print(f"  $$\\Delta_{step['n']} = {step['err']:.{precision}g}$$")

        if total_steps > 2:
            print("\n**Hai bước lặp cuối cùng:**")
            for i in range(max(2, total_steps - 2), total_steps):
                step = history[i]
                print(f"* **$n={step['n']}$:**")
                print(f"  $$x_{step['n']} = {step['x_prev']:.{precision}f} - \\frac{{{step['fx_prev']:.{precision}f}}}{{{step['dfx_prev']:.{precision}f}}} = {step['xn']:.{precision}f}$$")
                # Dùng định dạng .g cho sai số
                print(f"  $$\\Delta_{step['n']} = {step['err']:.{precision}g}$$")

        print("\n### 4. Bảng lặp rút gọn\n")
        print(f"| $n$ | $x_{{n-1}}$ | $f(x_{{n-1}})$ | $f'(x_{{n-1}})$ | $x_n$ | $\\Delta_n$ | ")
        print("| :--- | :--- | :--- | :--- | :--- | :--- | ")
        
        for i in range(min(3, total_steps)):
            s = history[i]
            # Cột sai số dùng định dạng .g
            print(f"| {s['n']} | {s['x_prev']:.{precision}f} | {s['fx_prev']:.{precision}f} | {s['dfx_prev']:.{precision}f} | {s['xn']:.{precision}f} | {s['err']:.{precision}g} |")
            
        if total_steps > 6:
            print("| ... | ... | ... | ... | ... | ... |")
            
        if total_steps > 3:
            for i in range(max(3, total_steps - 3), total_steps):
                s = history[i]
                # Cột sai số dùng định dạng .g
                print(f"| {s['n']} | {s['x_prev']:.{precision}f} | {s['fx_prev']:.{precision}f} | {s['dfx_prev']:.{precision}f} | {s['xn']:.{precision}f} | {s['err']:.{precision}g} |")

        print(f"\n=> KẾT LUẬN: $x \\approx {history[-1]['x_prev']:.{precision}f}$")

    except Exception as e:
        print(f"\n[!] CÓ LỖI XẢY RA TRONG QUÁ TRÌNH NHẬP LIỆU HOẶC TÍNH TOÁN:")
        print(f"Chi tiết lỗi: {e}")

if __name__ == "__main__":
    newton_method()
    input("\nNhấn Enter để thoát chương trình...")

=== CHƯƠNG TRÌNH GIẢI PHƯƠNG TRÌNH BẰNG PHƯƠNG PHÁP TIẾP TUYẾN ===
(Mẹo: Gõ hàm số cần ghi rõ dấu nhân, ví dụ: x**3 + 3*x**2 - 1)


### 1. Kiểm tra điều kiện và Chọn điểm xuất phát
* **Khoảng phân li nghiệm:** $f(0.1) = -0.9690000$, $f(1.0) = 3.0000000 \Rightarrow f(0.1)f(1.0) < 0$
* **Điều kiện đạo hàm:** $f'(x) = 3 x^{2} + 6 x$ và $f''(x) = 6 x + 6$ xác định dấu không đổi.
* **Chọn $x_0$:** Vì $f(1.0) \cdot f''(1.0) > 0$, ta chọn $x_0 = 1.0$.

### 2. Xây dựng công thức lặp và Đánh giá sai số
* **Các hằng số:** $m_1 = \min |f'(x)| = 0.6300000$, $M_2 = \max |f''(x)| = 12.0000000$
* **Công thức lặp:**
  $$x_n = x_{n-1} - \frac{f(x_{n-1})}{f'(x_{n-1})} = x_{n-1} - \frac{x^{3} + 3 x^{2} - 1}{3 x^{2} + 6 x}$$
* **Dự đoán số bước lặp (Tiên nghiệm):**
  Tính $q = \frac{M_2}{2m_1} |x_1 - x_0| = 3.174603$
  $\Rightarrow$ Do $q \ge 1$, không thể ước lượng số bước lặp bằng công thức tiên nghiệm cơ bản (nhưng thuật toán vẫn có thể hội tụ).
* **Điều kiện dừng thực tế (Hậu nghiệm):**
  $$\Delta_n =